In [14]:
import os

medquad_path = '../../MedQuAD'

folders = sorted(os.listdir(medquad_path))
for f in folders:
    full = os.path.join(medquad_path, f)
    if os.path.isdir(full):
        count = len([x for x in os.listdir(full) if x.endswith('.xml')])
        print(f"{f}: {count} files")

.git: 0 files
10_MPlus_ADAM_QA: 4366 files
11_MPlusDrugs_QA: 1312 files
12_MPlusHerbsSupplements_QA: 99 files
1_CancerGov_QA: 116 files
2_GARD_QA: 2685 files
3_GHR_QA: 1086 files
4_MPlus_Health_Topics_QA: 981 files
5_NIDDK_QA: 157 files
6_NINDS_QA: 277 files
7_SeniorHealth_QA: 48 files
8_NHLBI_QA_XML: 88 files
9_CDC_QA: 59 files


In [15]:
import xml.etree.ElementTree as ET
import json

def parse_medquad_xml(filepath):
    """Parse satu file XML MedQuAD, return list of QA pairs."""
    pairs = []
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
        
        # Ambil focus 
        focus_elem = root.find('.//Focus')
        focus = focus_elem.text.strip() if focus_elem is not None and focus_elem.text else ""
        
        # Cari semua QAPair
        for qapair in root.findall('.//QAPair'):
            question_elem = qapair.find('Question')
            answer_elem = qapair.find('Answer')
            
            if question_elem is not None and answer_elem is not None:
                q_text = question_elem.text or question_elem.get('text', '') or ''
                a_text = answer_elem.text or ''
                
                q_text = q_text.strip()
                a_text = a_text.strip()
                
                if q_text and a_text:
                    pairs.append({
                        'question': q_text,
                        'answer': a_text,
                        'focus': focus,
                        'source_file': os.path.basename(filepath),
                        'source': 'MedQuAD (NIH)'
                    })
    except Exception as e:
        print(f"Error parsing {filepath}: {e}")
    
    return pairs

# Keywords untuk filter diabetes-related content
diabetes_keywords = [
    'diabetes', 'diabetic', 'insulin', 'glucose', 'blood sugar',
    'hyperglycemia', 'hypoglycemia', 'a1c', 'hemoglobin a1c',
    'type 1 diabetes', 'type 2 diabetes', 'gestational diabetes',
    'prediabetes', 'pre-diabetes', 'metformin',
    'diabetic retinopathy', 'diabetic neuropathy', 'diabetic nephropathy',
    'diabetic ketoacidosis', 'pancreas', 'endocrine',
    'obesity', 'overweight', 'bmi', 'body mass index',
    'cholesterol', 'hypertension', 'high blood pressure',
    'cardiovascular', 'heart disease',
]

def is_diabetes_related(qa_pair):
    """Cek apakah QA pair terkait diabetes."""
    text = (qa_pair['question'] + ' ' + qa_pair['answer'] + ' ' + qa_pair['focus']).lower()
    return any(kw in text for kw in diabetes_keywords)

# Parse semua folder
all_pairs = []
diabetes_pairs = []

for folder in folders:
    folder_path = os.path.join(medquad_path, folder)
    if not os.path.isdir(folder_path):
        continue
    
    for filename in sorted(os.listdir(folder_path)):
        if not filename.endswith('.xml'):
            continue
        filepath = os.path.join(folder_path, filename)
        pairs = parse_medquad_xml(filepath)
        all_pairs.extend(pairs)
        
        for p in pairs:
            if is_diabetes_related(p):
                diabetes_pairs.append(p)

print(f"Total QA pairs parsed: {len(all_pairs)}")
print(f"Diabetes-related QA pairs: {len(diabetes_pairs)}")

Total QA pairs parsed: 16407
Diabetes-related QA pairs: 2394


In [16]:
# Simpan hasil filter
output_path = '../chatbot/data/medquad_diabetes.json'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(diabetes_pairs, f, indent=2, ensure_ascii=False)

print(f'Saved {len(diabetes_pairs)} diabetes QA pairs to {output_path}')

# Contoh
for i, pair in enumerate(diabetes_pairs[:3]):
    print(f'\n-- Example {i+1} --')
    print(f'Focus: {pair['focus']}')
    print(f'Q: {pair['question'][:100]}...')
    print(f'A: {pair['answer'][:150]}...')

Saved 2394 diabetes QA pairs to ../chatbot/data/medquad_diabetes.json

-- Example 1 --
Focus: Childhood Acute Myeloid Leukemia and Other Myeloid Malignancies
Q: What is the outlook for Childhood Acute Myeloid Leukemia and Other Myeloid Malignancies ?...
A: Certain factors affect prognosis (chance of recovery) and treatment options. The prognosis (chance of recovery) and treatment options for childhood AM...

-- Example 2 --
Focus: Adult Soft Tissue Sarcoma
Q: What are the stages of Adult Soft Tissue Sarcoma ?...
A: Key Points
                    - After adult soft tissue sarcoma has been diagnosed, tests are done to find out if cancer cells have spread within the...

-- Example 3 --
Focus: Gastrointestinal Stromal Tumors
Q: What are the stages of Gastrointestinal Stromal Tumors ?...
A: Key Points
                    - After a gastrointestinal stromal tumor has been diagnosed, tests are done to find out if cancer cells have spread wit...


In [17]:
# Load MedQuAD diabetes
with open('../chatbot/data/medquad_diabetes.json', 'r', encoding='utf-8') as f:
    medquad_data = json.load(f)

# Load stats QA dataset
stats_qa_path = '../chatbot/data/glucosense_stats_qa_dataset.json'
stats_data = []
if os.path.exists(stats_qa_path):
    with open(stats_qa_path, 'r', encoding='utf-8') as f:
        stats_data = json.load(f)
    print(f"Loaded {len(stats_data)} stats QA pairs")
else:
    print(f"File {stats_qa_path} not found — skipping stats QA")

# Gabungkan menjadi format dokumen yang seragam
corpus = []

# Dari MedQuAD: setiap jawaban jadi satu dokumen
for item in medquad_data:
    corpus.append({
        'text': f"Question: {item['question']}\n\nAnswer: {item['answer']}",
        'metadata': {
            'source': item.get('source', 'MedQuAD'),
            'focus': item.get('focus', ''),
            'type': 'medical_qa'
        }
    })

# Dari stats QA
for item in stats_data:
    q = item.get('question', item.get('Q', ''))
    a = item.get('answer', item.get('A', ''))
    if q and a:
        corpus.append({
            'text': f"Question: {q}\n\nAnswer: {a}",
            'metadata': {
                'source': 'GlucoSense Stats (BRFSS 2015)',
                'type': 'statistics_qa'
            }
        })

print(f"\nTotal corpus documents: {len(corpus)}")
print(f"- MedQuAD: {len(medquad_data)}")
print(f"- Stats QA: {len(stats_data)}")

File ../chatbot/data/glucosense_stats_qa_dataset.json not found — skipping stats QA

Total corpus documents: 2394
- MedQuAD: 2394
- Stats QA: 0


In [18]:
def chunk_text(text, max_words=300, overlap_words=50):
    words = text.split()
    if len(words) <= max_words:
        return [text]
    
    chunks = []
    start = 0
    while start < len(words):
        end = start + max_words
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start = end - overlap_words 
    
    return chunks

# Chunk semua dokumen
chunked_corpus = []
for doc in corpus:
    chunks = chunk_text(doc['text'])
    for i, chunk in enumerate(chunks):
        chunked_corpus.append({
            'text': chunk,
            'metadata': {
                **doc['metadata'],
                'chunk_index': i,
                'total_chunks': len(chunks)
            }
        })

print(f"After chunking: {len(chunked_corpus)} chunks (from {len(corpus)} documents)")
print(f"Average chunk length: {sum(len(c['text'].split()) for c in chunked_corpus) / len(chunked_corpus):.0f} words")

After chunking: 4337 chunks (from 2394 documents)
Average chunk length: 217 words


In [19]:
from sentence_transformers import SentenceTransformer
import chromadb

# 1. Load model MULTILINGUAL
print("Loading multilingual embedding model...")
embedder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
print("Model loaded")

# 2. Setup ChromaDB (persistent — data tersimpan di disk)
chroma_path = '../chatbot/data/chroma_db'
client = chromadb.PersistentClient(path=chroma_path)

# Hapus collection lama kalau ada (supaya bisa re-run)
try:
    client.delete_collection('glucosense_kb')
    print("Deleted old collection")
except:
    pass

collection = client.create_collection(
    name='glucosense_kb',
    metadata={"hnsw:space": "cosine"}  # pakai cosine similarity
)
print("ChromaDB collection created")

# 3. Embed dan simpan semua chunks
batch_size = 100
total = len(chunked_corpus)

for i in range(0, total, batch_size):
    batch = chunked_corpus[i:i+batch_size]
    
    texts = [item['text'] for item in batch]
    metadatas = [item['metadata'] for item in batch]
    ids = [f"doc_{i+j}" for j in range(len(batch))]
    
    # Embed
    embeddings = embedder.encode(texts).tolist()
    
    # Simpan ke Chroma
    collection.add(
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )
    
    print(f"  Embedded {min(i+batch_size, total)}/{total} chunks...")

print(f"\nAll {total} chunks embedded and stored in ChromaDB")
print(f"Database path: {chroma_path}")

Loading multilingual embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4411.23it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded
Deleted old collection
ChromaDB collection created
  Embedded 100/4337 chunks...
  Embedded 200/4337 chunks...
  Embedded 300/4337 chunks...
  Embedded 400/4337 chunks...
  Embedded 500/4337 chunks...
  Embedded 600/4337 chunks...
  Embedded 700/4337 chunks...
  Embedded 800/4337 chunks...
  Embedded 900/4337 chunks...
  Embedded 1000/4337 chunks...
  Embedded 1100/4337 chunks...
  Embedded 1200/4337 chunks...
  Embedded 1300/4337 chunks...
  Embedded 1400/4337 chunks...
  Embedded 1500/4337 chunks...
  Embedded 1600/4337 chunks...
  Embedded 1700/4337 chunks...
  Embedded 1800/4337 chunks...
  Embedded 1900/4337 chunks...
  Embedded 2000/4337 chunks...
  Embedded 2100/4337 chunks...
  Embedded 2200/4337 chunks...
  Embedded 2300/4337 chunks...
  Embedded 2400/4337 chunks...
  Embedded 2500/4337 chunks...
  Embedded 2600/4337 chunks...
  Embedded 2700/4337 chunks...
  Embedded 2800/4337 chunks...
  Embedded 2900/4337 chunks...
  Embedded 3000/4337 chunks...
  Embedded 3100

In [22]:
# Test RAG pipeline langsung
import sys
sys.path.insert(0, '..')

from chatbot.rag import get_response

# Test 1: Pertanyaan bahasa Inggris
result = get_response("What are the risk factors for type 2 diabetes?")
print(f"Q: {result['query']}")
print(f"\nA: {result['answer']}")
print(f"\nSources: {len(result['sources'])}")
for s in result['sources']:
    print(f"  - {s['source']} (relevance: {s['relevance']})")

# Test 2: Pertanyaan bahasa Indonesia
result2 = get_response("Apa gejala diabetes tipe 2?")
print(f"Q: {result2['query']}")
print(f"\nA: {result2['answer']}")

# Test 3: Off-topic (harus ditolak)
result3 = get_response("Bagaimana cara masak nasi goreng?")
print(f"Q: {result3['query']}")
print(f"\nA: {result3['answer']}")

# Test 4: Pertanyaan obat spesifik (harus diarahkan ke dokter)
result4 = get_response("Berapa dosis metformin yang tepat untuk saya?")
print(f"Q: {result4['query']}")
print(f"\nA: {result4['answer']}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3339.80it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
d:\008 PROJECTS\glucosense\GlucoSense\notebooks\..\chatbot\rag.py:139: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
LLM call failed: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/

Q: What are the risk factors for type 2 diabetes?

A: Sorry, I encountered an error while processing your question. Please try again later.

Sources: 5
  - MedQuAD (NIH) (relevance: 0.861)
  - MedQuAD (NIH) (relevance: 0.807)
  - MedQuAD (NIH) (relevance: 0.805)
  - MedQuAD (NIH) (relevance: 0.795)
  - MedQuAD (NIH) (relevance: 0.794)


LLM call failed: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 22.598131508s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-FreeT

Q: Apa gejala diabetes tipe 2?

A: Sorry, I encountered an error while processing your question. Please try again later.


LLM call failed: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 22.237374878s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-FreeT

Q: Bagaimana cara masak nasi goreng?

A: Sorry, I encountered an error while processing your question. Please try again later.


LLM call failed: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 21.776273576s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota

Q: Berapa dosis metformin yang tepat untuk saya?

A: Sorry, I encountered an error while processing your question. Please try again later.
